# Tutorial: Flujo Notebook → SQLite + CSVs

Este notebook demuestra el flujo estándar para procesar datos y registrar resultados en el sistema de memorias.

**Flujo:**
1. Crear una memoria (experimento) en SQLite
2. Procesar archivos HDF5 (leer curvas)
3. Guardar resultados como CSVs en `data/processed/`
4. Registrar análisis en SQLite
5. Referenciar fotos y otros archivos
6. El dashboard lee SQLite + CSVs para visualización

In [ ]:
import os, sys
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

PROJECT_ROOT = Path(os.path.abspath(os.path.join(os.getcwd(), '../..')))
sys.path.insert(0, str(PROJECT_ROOT))

from src import db
from src.readers.keithley_dean import list_measurements, read_curve, read_all_curves
from src.analysis import extract_vth_for_transfers

db.init_db()
print('Database initialized at', db.DB_PATH)
print('Project root:', PROJECT_ROOT)

## 1. Crear una memoria (experimento)

Primero creamos una memoria en SQLite para documentar este experimento.

In [ ]:
# Crear memoria de tipo experiment (batch de dispositivos)
memory_id = db.create_memory(
    title="Batch 3 Kapton - Oct 2026",
    type="experiment",
    subtype="batch",
    date="2026-10-02",
    description="Medición de dispositivos Kapton con proceso BAMS + PFBT",
    substrate="Kapton",
    process="BAMS + PFBT",
    template_id=2,  # kapton_batch template
    custom_fields={
        "thickness": "125um",
        "SAM_concentration": "1mM",
        "bar_number": "3",
    },
    references={
        "hdf5": ["data/raw/2026-10-02_test.hdf5"],
    }
)

print(f'Created memory with id={memory_id}')

# Verificar que se creó correctamente
memory = db.get_memory(memory_id)
print(f'\nTitle: {memory["title"]}')
print(f'Type: {memory["type"]} ({memory["subtype"]})')
print(f'Date: {memory["date"]}')
print(f'Substrate: {memory["substrate"]}')
print(f'Process: {memory["process"]}')
print(f'\nCustom fields: {len(memory["fields"])}')
for f in memory["fields"]:
    print(f'  - {f["field_name"]}: {f["field_value"]}')
print(f'\nReferences: {len(memory["references"])}')
for r in memory["references"]:
    print(f'  - {r["ref_type"]}: {r["ref_path"]}')

## 2. Procesar HDF5 y extraer parámetros

Ahora procesamos el archivo HDF5 para extraer Vth de las curvas de transferencia.

In [ ]:
# Cargar archivo HDF5
hdf5_path = PROJECT_ROOT / "data/raw/2026-10-02_test.hdf5"

# Listar mediciones de tipo transfer
measurement_list_dic = list_measurements(hdf5_path)
transfer_measurements = [m["name"] for m in measurement_list_dic if m["mode"] == "transfer"]

print(f'Found {len(transfer_measurements)} transfer measurements')
for m in transfer_measurements[:5]:  # Mostrar primeros 5
    print(f'  - {m}')

In [ ]:
# Extraer Vth para todas las mediciones transfer
df_vth = extract_vth_for_transfers(
    hdf5_path,
    transfer_measurements,
    v_min_range=(-0.40, -0.38),
    v_max_range=(-0.06, -0.00),
)

print(f'Extracted Vth for {len(df_vth)} measurements')
print('\nFirst 5 results:')
display(df_vth.head())

## 3. Guardar resultados en CSV

Los resultados se guardan en `data/processed/` para que el dashboard los pueda leer.

In [ ]:
# Guardar CSV en data/processed/
processed_dir = PROJECT_ROOT / "data" / "processed"
processed_dir.mkdir(parents=True, exist_ok=True)

csv_filename = f"memory_{memory_id}_vth_results.csv"
csv_path = processed_dir / csv_filename

df_vth.to_csv(csv_path, index=False)
print(f'Saved CSV to: {csv_path}')

# Registrar referencia al CSV en la memoria
db.add_reference(memory_id, "csv", f"data/processed/{csv_filename}")
print('Added CSV reference to memory')

## 4. Registrar análisis en SQLite

Guardamos los parámetros y resultados del análisis en SQLite para persistencia.

In [ ]:
# Registrar análisis en SQLite
analysis_id = db.save_analysis_result(
    memory_id=memory_id,
    function_name="extract_vth_for_transfers",
    params={
        "v_min_range": [-0.40, -0.38],
        "v_max_range": [-0.06, -0.00],
    },
    results_csv_path=f"data/processed/{csv_filename}",
)

print(f'Saved analysis result with id={analysis_id}')

# Ver análisis guardados
memory_updated = db.get_memory(memory_id)
print(f'\nAnalysis results: {len(memory_updated["analysis_results"])}')
for a in memory_updated["analysis_results"]:
    print(f'  - {a["function_name"]} (id={a["id"]})')
    print(f'    Params: {a["params_json"]}')
    print(f'    CSV: {a["results_csv_path"]}')

## 5. Agregar fotos (opcional)

Si tienes fotos del experimento, puedes referenciarlas.

In [ ]:
# Buscar fotos disponibles
photos_dir = PROJECT_ROOT / "data" / "photos" / "kapton"
if photos_dir.exists():
    photos = list(photos_dir.rglob("*.jpg"))[:3]  # Primeras 3
    print(f'Found {len(photos)} photos in kapton/')
    for p in photos:
        rel_path = p.relative_to(PROJECT_ROOT)
        print(f'  - {rel_path}')
        db.add_reference(memory_id, "photo", str(rel_path))
    print(f'\nAdded {len(photos)} photo references to memory')
else:
    print('No photos directory found')

## 6. Resumen: Cómo el dashboard lee los datos

El dashboard (NiceGUI) hace lo siguiente:
1. Lee SQLite para listar memorias (`db.list_memories()`)
2. Para cada memoria, carga referencias (HDF5, fotos, CSVs)
3. Lee los CSVs de `data/processed/` para graficar
4. Muestra fotos con zoom
5. Permite comparación entre memorias

In [ ]:
# Ejemplo de cómo el dashboard lee los datos
print('=== Dashboard preview ===\n')

# Listar todas las memorias
all_memories = db.list_memories()
print(f'Total memories: {len(all_memories)}')

# Mostrar detalles de la memoria creada
for m in all_memories:
    print(f'\n[{m["id"]}] {m["title"]}')
    print(f'    Type: {m["type"]} ({m.get("subtype", "N/A")})')
    print(f'    Date: {m["date"]}')
    print(f'    Substrate: {m["substrate"]}')
    
    # Cargar detalles
    memory_detail = db.get_memory(m["id"])
    
    # Mostrar referencias
    n_refs = len(memory_detail["references"])
    print(f'    References: {n_refs} files')
    for r in memory_detail["references"]:
        print(f'      - {r["ref_type"]}: {r["ref_path"]}')
    
    # Mostrar análisis
    n_analysis = len(memory_detail["analysis_results"])
    print(f'    Analysis: {n_analysis} results')

## 7. Listar y filtrar memorias

El dashboard puede filtrar memorias por tipo, sustrato, fecha, etc.

In [ ]:
# Filtrar por tipo
experiments = db.list_memories(type="experiment")
print(f'Experiments: {len(experiments)}')

# Filtrar por sustrato
kapton_memories = db.list_memories(substrate="Kapton")
print(f'Kapton memories: {len(kapton_memories)}')

# Filtrar por rango de fechas
oct_memories = db.list_memories(date_from="2026-10-01", date_to="2026-10-31")
print(f'October 2026 memories: {len(oct_memories)}')

# Búsqueda por texto
batch3_memories = db.list_memories(search="Batch 3")
print(f'Memories with "Batch 3": {len(batch3_memories)}')

## 8. Templates

Los templates definen campos personalizados predefinidos para diferentes tipos de experimentos.

In [ ]:
# Listar templates disponibles
templates = db.list_templates()
print(f'Available templates: {len(templates)}\n')

for t in templates:
    print(f'[{t["id"]}] {t["name"]}')
    print(f'    {t["description"]}')
    import json
    fields = json.loads(t["fields_json"])
    print(f'    Fields: {len(fields)}')
    for f in fields:
        print(f'      - {f["name"]} ({f["type"]}): default={f["default"]}')
    print()

## Resumen

Este notebook demuestra el flujo completo:
1. **Crear memoria**: `db.create_memory()` con metadatos base + custom fields
2. **Procesar HDF5**: usar `src.readers` y `src.analysis`
3. **Guardar CSV**: en `data/processed/` con nombre descriptivo
4. **Registrar análisis**: `db.save_analysis_result()` con parámetros y path al CSV
5. **Referenciar archivos**: `db.add_reference()` para fotos, HDF5s, etc.

**Próximos pasos:**
- El dashboard (NiceGUI) lee estos datos para visualización
- Puedes crear más notebooks siguiendo este patrón
- Los templates en `src/schema.yaml` definen campos estándar